# Phase 6: Execution, Market Microstructure & Systems  
## Notebook 06_09 — Limit Order Fill Probability Model

### Research question

How can we estimate the probability that a passive limit order fills before expiry, using top-of-book state, queue-position proxies, imbalance, latency, volatility, spread, and time-in-force?

This notebook follows:

```text
06_01_almgren_chriss_optimal_execution
06_02_dynamic_programming_execution
06_03_square_root_market_impact_model
06_04_hawkes_process_order_flow
06_05_rough_volatility_estimation
06_06_microstructure_friction_cpp_core
06_07_limit_board_margin_and_deadband_rebalancing
06_08_avellaneda_stoikov_market_making
06_09_limit_order_fill_probability_model
```

The previous notebook used Avellaneda-Stoikov to generate market-making quotes. But a quote is only useful if it has a credible fill model.

This notebook answers:

> Given a candidate limit order, what is the probability it fills before expiry, and is resting passively better than crossing the spread?

It covers:

1. limit order fill mechanics;
2. queue-position intuition;
3. top-of-book features;
4. distance-to-touch;
5. time-in-force;
6. latency;
7. imbalance;
8. volatility and spread regime;
9. synthetic L1/L2-like event stream;
10. limit-order simulation;
11. fill labels and fill time;
12. logistic fill model;
13. hazard-style fill model;
14. calibration curves;
15. Brier score and log loss;
16. AUC and ranking diagnostics;
17. feature importance;
18. probability by bucket;
19. passive versus marketable expected value;
20. adverse selection after fill;
21. cancellation threshold policy;
22. latency sensitivity;
23. governance flags;
24. saved outputs and manifest.

The key idea:

> A passive order is not “free spread capture”; it is a probabilistic bet on queue position, future order flow, adverse selection, and time.

## 1. Limit order fill probability

A passive buy limit order fills if enough sell marketable flow reaches its price before the order expires.

A passive sell limit order fills if enough buy marketable flow reaches its price before expiry.

For a simplified top-of-book queue:

$$
P(fill) = P(\text{future contra flow} \geq queueAhead + orderSize)
$$

This depends on:

- queue ahead;
- displayed depth;
- order size;
- time-in-force;
- spread;
- price distance from touch;
- order-flow imbalance;
- volatility;
- latency;
- cancellation/replacement behaviour.

The model is probabilistic because future order flow is uncertain.

## 2. Feature intuition

For a buy limit order:

### Higher fill probability

- order price closer to ask;
- smaller queue ahead;
- smaller order size;
- higher sell flow intensity;
- negative order-flow imbalance;
- wider time-in-force;
- higher volatility.

### Lower fill probability

- order far from touch;
- large queue ahead;
- low contra flow;
- high same-side queue;
- short time-in-force;
- stale quote due to latency.

Fill probability alone is not enough. We also need expected adverse selection:

$$
\begin{aligned}
EV &= P(fill)\times expectedEdge \\
&\quad - P(fill)\times adverseSelection \\
&\quad - opportunityCost
\end{aligned}
$$

## 3. Model choices

This notebook implements two practical models:

### Logistic fill model

$$
P(fill=1|x) = \frac{1}{1+\exp(-x^\top\beta)}
$$

Good for fixed-horizon fill classification.

### Discrete hazard model

For each order and each time bucket:

$$
P(fill\ at\ bucket\ k | not\ filled\ before, x_k)
$$

This is more natural for time-to-fill, but more complex.

We implement both in compact form.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

SKLEARN_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class LimitOrderFillConfig:
    n_events: int = 120_000
    n_orders: int = 12_000
    seed: int = 42
    initial_mid: float = 100.0
    tick_size: float = 0.01
    base_spread_ticks: int = 2
    max_spread_ticks: int = 10
    base_depth: float = 2_500.0
    min_time_in_force_events: int = 5
    max_time_in_force_events: int = 80
    max_latency_events: int = 8
    passive_rebate_bps: float = -0.05
    taker_fee_bps: float = 0.45
    adverse_selection_horizon: int = 20
    order_size_median: float = 650.0
    calibration_train_frac: float = 0.70
    probability_threshold: float = 0.55
    output_subdir: str = "limit_order_fill_probability_model_v1"

config = LimitOrderFillConfig()
config

## 5. Simulate L1/L2-like event stream

We simulate:

- bid;
- ask;
- mid;
- spread;
- bid and ask depth;
- order-flow signs;
- buy and sell marketable volume;
- volatility regime;
- imbalance.

This is not full order-book data. It is a compact top-of-book-plus-flow approximation.

In [ ]:
def round_to_tick(x, tick):
    return np.round(x / tick) * tick

def simulate_market_stream(config: LimitOrderFillConfig):
    rng = np.random.default_rng(config.seed)
    n = config.n_events
    event_id = np.arange(n)

    regime = np.zeros(n, dtype=int)
    for t in range(1, n):
        if regime[t - 1] == 0:
            regime[t] = rng.choice([0, 1], p=[0.992, 0.008])
        else:
            regime[t] = rng.choice([0, 1], p=[0.045, 0.955])

    vol_scale = np.where(regime == 0, 1.0, 2.5)
    returns = rng.standard_t(df=6, size=n) * 0.00010 * vol_scale
    shock_idx = rng.choice(n, size=max(5, n // 7000), replace=False)
    returns[shock_idx] += rng.normal(0.0, 0.0025, size=len(shock_idx))

    mid = config.initial_mid * np.exp(np.cumsum(returns))
    mid = round_to_tick(mid, config.tick_size)

    spread_ticks = rng.choice(
        np.arange(1, config.max_spread_ticks + 1),
        size=n,
        p=np.array([0.35, 0.25, 0.14, 0.08, 0.055, 0.04, 0.035, 0.03, 0.025, 0.01]),
    )
    spread_ticks = spread_ticks + regime * rng.integers(0, 4, size=n)
    spread_ticks = np.clip(spread_ticks, 1, config.max_spread_ticks)

    spread = spread_ticks * config.tick_size
    bid = round_to_tick(mid - spread / 2, config.tick_size)
    ask = round_to_tick(mid + spread / 2, config.tick_size)
    ask = np.maximum(ask, bid + config.tick_size)

    bid_depth = rng.lognormal(np.log(config.base_depth), 0.55, size=n) * np.where(regime == 0, 1.0, 0.45)
    ask_depth = rng.lognormal(np.log(config.base_depth), 0.55, size=n) * np.where(regime == 0, 1.0, 0.45)

    # Order-flow imbalance state.
    latent_flow = np.zeros(n)
    phi = 0.96
    for t in range(1, n):
        latent_flow[t] = phi * latent_flow[t - 1] + rng.normal(0, 0.18)
        latent_flow[t] += 0.25 * returns[t] / max(np.std(returns[:max(t, 2)]), 1e-8)

    flow_prob_buy = 1.0 / (1.0 + np.exp(-latent_flow))
    flow_sign = np.where(rng.uniform(size=n) < flow_prob_buy, 1, -1)

    buy_mkt_volume = np.where(
        flow_sign > 0,
        rng.lognormal(np.log(600), 0.75, size=n) * vol_scale,
        rng.lognormal(np.log(90), 0.60, size=n),
    )
    sell_mkt_volume = np.where(
        flow_sign < 0,
        rng.lognormal(np.log(600), 0.75, size=n) * vol_scale,
        rng.lognormal(np.log(90), 0.60, size=n),
    )

    stream = pd.DataFrame({
        "event_id": event_id,
        "regime": regime,
        "mid": mid,
        "bid": bid,
        "ask": ask,
        "spread": ask - bid,
        "spread_ticks": (ask - bid) / config.tick_size,
        "bid_depth": bid_depth,
        "ask_depth": ask_depth,
        "flow_sign": flow_sign,
        "buy_mkt_volume": buy_mkt_volume,
        "sell_mkt_volume": sell_mkt_volume,
        "latent_flow": latent_flow,
    })

    stream["book_imbalance"] = (stream["bid_depth"] - stream["ask_depth"]) / (stream["bid_depth"] + stream["ask_depth"])
    stream["flow_imbalance_50"] = (
        stream["buy_mkt_volume"].rolling(50).sum()
        - stream["sell_mkt_volume"].rolling(50).sum()
    ) / (
        stream["buy_mkt_volume"].rolling(50).sum()
        + stream["sell_mkt_volume"].rolling(50).sum()
    )
    stream["flow_imbalance_50"] = stream["flow_imbalance_50"].fillna(0.0)
    stream["realised_vol_100"] = stream["mid"].pct_change().rolling(100).std().fillna(method="bfill").fillna(0.0)
    stream["mid_return_fwd"] = stream["mid"].pct_change().shift(-1).fillna(0.0)

    return stream

stream = simulate_market_stream(config)

stream.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(stream["event_id"], stream["mid"])
plt.title("Synthetic Mid Price")
plt.xlabel("Event")
plt.ylabel("Mid")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(stream["event_id"], stream["book_imbalance"], label="book imbalance", alpha=0.7)
plt.plot(stream["event_id"], stream["flow_imbalance_50"], label="flow imbalance 50", alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Book and Flow Imbalance")
plt.xlabel("Event")
plt.legend()
plt.show()

## 6. Generate candidate limit orders

Each candidate order has:

- side;
- submission event;
- latency;
- time-in-force;
- size;
- limit price;
- queue ahead;
- distance to touch.

We intentionally generate orders at different distances from touch so the fill model sees a range of probabilities.

In [ ]:
def generate_candidate_limit_orders(stream, config):
    rng = np.random.default_rng(config.seed + 123)
    n_orders = config.n_orders

    submit_event = np.sort(
        rng.choice(
            np.arange(0, config.n_events - config.max_time_in_force_events - config.max_latency_events - 5),
            size=n_orders,
            replace=False,
        )
    )

    side = rng.choice(np.array([1, -1], dtype=int), size=n_orders)
    size = rng.lognormal(np.log(config.order_size_median), 0.70, size=n_orders)
    size = np.maximum(1.0, np.round(size))

    time_in_force = rng.integers(
        config.min_time_in_force_events,
        config.max_time_in_force_events + 1,
        size=n_orders,
    )

    latency_events = rng.integers(0, config.max_latency_events + 1, size=n_orders)

    distance_ticks = rng.choice(
        np.array([0, 1, 2, 3, 5, 8]),
        size=n_orders,
        p=np.array([0.30, 0.26, 0.18, 0.12, 0.09, 0.05]),
    )

    limit_price = np.zeros(n_orders)
    arrival_mid = stream.loc[submit_event, "mid"].to_numpy()
    arrival_bid = stream.loc[submit_event, "bid"].to_numpy()
    arrival_ask = stream.loc[submit_event, "ask"].to_numpy()

    for i in range(n_orders):
        if side[i] > 0:
            # Buy limit at bid minus distance ticks.
            limit_price[i] = arrival_bid[i] - distance_ticks[i] * config.tick_size
            limit_price[i] = np.floor(limit_price[i] / config.tick_size) * config.tick_size
        else:
            # Sell limit at ask plus distance ticks.
            limit_price[i] = arrival_ask[i] + distance_ticks[i] * config.tick_size
            limit_price[i] = np.ceil(limit_price[i] / config.tick_size) * config.tick_size

    orders = pd.DataFrame({
        "order_id": np.arange(n_orders, dtype=int),
        "submit_event": submit_event,
        "side": side,
        "side_name": np.where(side > 0, "BUY", "SELL"),
        "size": size,
        "time_in_force": time_in_force,
        "latency_events": latency_events,
        "active_event": submit_event + latency_events,
        "expiry_event": submit_event + latency_events + time_in_force,
        "distance_ticks": distance_ticks,
        "limit_price": limit_price,
        "arrival_mid": arrival_mid,
        "arrival_bid": arrival_bid,
        "arrival_ask": arrival_ask,
    })

    return orders

orders = generate_candidate_limit_orders(stream, config)

orders.head()

## 7. Simulate limit-order outcomes

For each order, we scan future events until expiry.

A buy limit fills when:

$$
ask_t \le limitPrice
$$

or enough sell marketable volume consumes queue ahead.

A sell limit fills when:

$$
bid_t \ge limitPrice
$$

or enough buy marketable volume consumes queue ahead.

The queue-ahead proxy is based on displayed same-side depth and the order's relative position.

In [ ]:
def simulate_limit_order_outcomes(stream, orders, config):
    rows = []

    for _, order in orders.iterrows():
        side = int(order["side"])
        active = int(order["active_event"])
        expiry = min(int(order["expiry_event"]), len(stream) - 1)
        limit_price = float(order["limit_price"])
        order_size = float(order["size"])

        active_row = stream.iloc[active]
        if side > 0:
            same_depth = active_row["bid_depth"]
            contra_col = "sell_mkt_volume"
            touch_price = active_row["bid"]
            price_distance_ticks = max(round((touch_price - limit_price) / config.tick_size), 0)
        else:
            same_depth = active_row["ask_depth"]
            contra_col = "buy_mkt_volume"
            touch_price = active_row["ask"]
            price_distance_ticks = max(round((limit_price - touch_price) / config.tick_size), 0)

        queue_ahead = same_depth * (1.0 + 0.45 * price_distance_ticks)
        queue_needed = queue_ahead + order_size

        cumulative_contra = 0.0
        filled = False
        fill_event = -1
        fill_price = np.nan
        fill_qty = 0.0
        fill_reason = "NO_FILL"

        for ev in range(active, expiry + 1):
            row = stream.iloc[ev]

            if side > 0:
                price_cross = row["ask"] <= limit_price
            else:
                price_cross = row["bid"] >= limit_price

            cumulative_contra += row[contra_col]

            if price_cross:
                filled = True
                fill_event = ev
                fill_price = limit_price
                fill_qty = order_size
                fill_reason = "PRICE_CROSS"
                break

            if cumulative_contra >= queue_needed:
                filled = True
                fill_event = ev
                fill_price = limit_price
                fill_qty = order_size
                fill_reason = "QUEUE_DEPLETION"
                break

        if filled:
            horizon_event = min(fill_event + config.adverse_selection_horizon, len(stream) - 1)
            future_mid = stream.iloc[horizon_event]["mid"]
            fill_mid = stream.iloc[fill_event]["mid"]

            if side > 0:
                spread_capture_bps = (fill_mid - fill_price) / fill_mid * 10000.0
                adverse_selection_bps = (future_mid - fill_mid) / fill_mid * 10000.0
            else:
                spread_capture_bps = (fill_price - fill_mid) / fill_mid * 10000.0
                adverse_selection_bps = (fill_mid - future_mid) / fill_mid * 10000.0

            net_edge_bps = spread_capture_bps - adverse_selection_bps - config.passive_rebate_bps
            fill_time_events = fill_event - active
        else:
            spread_capture_bps = np.nan
            adverse_selection_bps = np.nan
            net_edge_bps = np.nan
            fill_time_events = np.nan

        rows.append({
            "order_id": int(order["order_id"]),
            "active_event": active,
            "expiry_event": expiry,
            "filled": int(filled),
            "fill_event": fill_event,
            "fill_time_events": fill_time_events,
            "fill_price": fill_price,
            "fill_qty": fill_qty,
            "fill_reason": fill_reason,
            "queue_ahead": queue_ahead,
            "queue_needed": queue_needed,
            "cumulative_contra_volume": cumulative_contra,
            "active_bid": active_row["bid"],
            "active_ask": active_row["ask"],
            "active_mid": active_row["mid"],
            "active_spread_ticks": active_row["spread_ticks"],
            "active_book_imbalance": active_row["book_imbalance"],
            "active_flow_imbalance_50": active_row["flow_imbalance_50"],
            "active_realised_vol_100": active_row["realised_vol_100"],
            "active_regime": active_row["regime"],
            "spread_capture_bps": spread_capture_bps,
            "adverse_selection_bps": adverse_selection_bps,
            "net_edge_bps": net_edge_bps,
        })

    outcomes = pd.DataFrame(rows)
    return outcomes

outcomes = simulate_limit_order_outcomes(stream, orders, config)

outcomes.head(), outcomes["filled"].mean()

## 8. Build modelling dataset

Features should be known at active order time.

Candidate features:

- side;
- order size relative to depth;
- queue ahead;
- distance ticks;
- spread ticks;
- book imbalance;
- flow imbalance;
- volatility;
- regime;
- latency;
- time-in-force.

In [ ]:
def build_model_dataset(orders, outcomes):
    df = orders.merge(outcomes, on="order_id", how="inner")

    df["side_buy"] = (df["side"] > 0).astype(int)
    df["size_to_depth"] = df["size"] / (df["queue_ahead"] + 1.0)
    df["queue_ahead_log"] = np.log1p(df["queue_ahead"])
    df["size_log"] = np.log1p(df["size"])
    df["tif_log"] = np.log1p(df["time_in_force"])
    df["latency_log"] = np.log1p(df["latency_events"])
    df["distance_ticks_log"] = np.log1p(df["distance_ticks"])
    df["queue_needed_log"] = np.log1p(df["queue_needed"])
    df["contra_volume_to_queue"] = df["cumulative_contra_volume"] / (df["queue_needed"] + 1.0)

    # Side-adjusted imbalance. Positive should generally mean more favourable contra pressure for fill.
    # For buy orders, sell pressure is favourable, so negative flow imbalance helps. For sell orders, positive helps.
    df["side_adjusted_flow_imbalance"] = np.where(df["side"] > 0, -df["active_flow_imbalance_50"], df["active_flow_imbalance_50"])
    df["side_adjusted_book_imbalance"] = np.where(df["side"] > 0, -df["active_book_imbalance"], df["active_book_imbalance"])

    feature_cols = [
        "side_buy",
        "size_log",
        "size_to_depth",
        "queue_ahead_log",
        "queue_needed_log",
        "distance_ticks",
        "distance_ticks_log",
        "time_in_force",
        "tif_log",
        "latency_events",
        "latency_log",
        "active_spread_ticks",
        "side_adjusted_book_imbalance",
        "side_adjusted_flow_imbalance",
        "active_realised_vol_100",
        "active_regime",
    ]

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_cols + ["filled"])

    return df, feature_cols

model_df, feature_cols = build_model_dataset(orders, outcomes)

model_df[feature_cols + ["filled"]].head()

## 9. Fill-rate diagnostics by bucket

In [ ]:
bucket_reports = []

for col in ["distance_ticks", "time_in_force", "latency_events", "active_spread_ticks", "active_regime"]:
    report = (
        model_df
        .groupby(col)
        .agg(
            n=("filled", "count"),
            fill_rate=("filled", "mean"),
            mean_queue=("queue_ahead", "mean"),
            mean_size=("size", "mean"),
        )
        .reset_index()
    )
    report["bucket_variable"] = col
    bucket_reports.append(report)

fill_bucket_report = pd.concat(bucket_reports, ignore_index=True)

fill_bucket_report.head(20)

In [ ]:
distance_report = fill_bucket_report[fill_bucket_report["bucket_variable"] == "distance_ticks"]

plt.figure(figsize=(9, 5))
plt.plot(distance_report["distance_ticks"], distance_report["fill_rate"], marker="o")
plt.title("Fill Rate by Distance from Touch")
plt.xlabel("Distance ticks")
plt.ylabel("Fill rate")
plt.show()

tif_report = fill_bucket_report[fill_bucket_report["bucket_variable"] == "time_in_force"]
plt.figure(figsize=(9, 5))
plt.scatter(tif_report["time_in_force"], tif_report["fill_rate"], s=np.clip(tif_report["n"], 5, 80))
plt.title("Fill Rate by Time-in-Force")
plt.xlabel("Time-in-force events")
plt.ylabel("Fill rate")
plt.show()

## 10. Train/test split

Use chronological split by active event.

This avoids training on future market conditions.

In [ ]:
def chronological_train_test(df, train_frac):
    df_sorted = df.sort_values("active_event").reset_index(drop=True)
    split = int(len(df_sorted) * train_frac)
    return df_sorted.iloc[:split].copy(), df_sorted.iloc[split:].copy()

train_df, test_df = chronological_train_test(model_df, config.calibration_train_frac)

pd.Series({
    "n_train": len(train_df),
    "n_test": len(test_df),
    "train_fill_rate": train_df["filled"].mean(),
    "test_fill_rate": test_df["filled"].mean(),
    "train_active_event_max": train_df["active_event"].max(),
    "test_active_event_min": test_df["active_event"].min(),
})

## 11. Logistic fill probability model

The model estimates:

$$
P(fill=1|x) = \frac{1}{1+\exp(-x^\top\beta)}
$$

We use scikit-learn if available and a simple fallback logistic regression otherwise.

In [ ]:
class SimpleLogistic:
    def __init__(self, lr=0.05, l2=1.0, n_iter=700):
        self.lr = lr
        self.l2 = l2
        self.n_iter = n_iter
        self.mean_ = None
        self.std_ = None
        self.coef_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1.0
        Xs = (X - self.mean_) / self.std_
        Xb = np.column_stack([np.ones(len(Xs)), Xs])
        w = np.zeros(Xb.shape[1])

        for _ in range(self.n_iter):
            z = Xb @ w
            p = 1.0 / (1.0 + np.exp(-np.clip(z, -35, 35)))
            grad = Xb.T @ (p - y) / len(y)
            grad[1:] += self.l2 * w[1:] / len(y)
            w -= self.lr * grad

        self.coef_ = w
        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        Xs = (X - self.mean_) / self.std_
        Xb = np.column_stack([np.ones(len(Xs)), Xs])
        z = Xb @ self.coef_
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -35, 35)))
        return np.column_stack([1 - p, p])

def make_logistic_model():
    if SKLEARN_AVAILABLE:
        return Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, C=0.7, solver="lbfgs")),
        ])
    return SimpleLogistic()

X_train = train_df[feature_cols].to_numpy()
y_train = train_df["filled"].to_numpy()
X_test = test_df[feature_cols].to_numpy()
y_test = test_df["filled"].to_numpy()

logistic_model = make_logistic_model()
logistic_model.fit(X_train, y_train)

test_prob = logistic_model.predict_proba(X_test)[:, 1]
train_prob = logistic_model.predict_proba(X_train)[:, 1]

test_predictions = test_df[["order_id", "active_event", "filled", "fill_time_events", "net_edge_bps"]].copy()
test_predictions["pred_fill_prob"] = test_prob

test_predictions.head()

## 12. Model metrics

Metrics:

- AUC: ranking quality;
- Brier score: probability accuracy;
- log loss: probabilistic likelihood;
- calibration by probability bucket.

In [ ]:
def safe_auc(y, p):
    if len(np.unique(y)) < 2:
        return np.nan
    if SKLEARN_AVAILABLE:
        return float(roc_auc_score(y, p))
    order = np.argsort(p)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(p) + 1)
    pos = y == 1
    n_pos = pos.sum()
    n_neg = len(y) - n_pos
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def safe_log_loss(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())

def brier(y, p):
    return float(np.mean((y - p) ** 2))

model_metrics = pd.DataFrame([
    {
        "sample": "train",
        "n": len(y_train),
        "fill_rate": y_train.mean(),
        "auc": safe_auc(y_train, train_prob),
        "brier": brier(y_train, train_prob),
        "log_loss": safe_log_loss(y_train, train_prob),
        "mean_pred_prob": train_prob.mean(),
    },
    {
        "sample": "test",
        "n": len(y_test),
        "fill_rate": y_test.mean(),
        "auc": safe_auc(y_test, test_prob),
        "brier": brier(y_test, test_prob),
        "log_loss": safe_log_loss(y_test, test_prob),
        "mean_pred_prob": test_prob.mean(),
    },
])

model_metrics

## 13. Calibration curve

A calibrated model should satisfy:

$$
P(fill|predicted\ probability \approx p) \approx p
$$

We bucket predictions into deciles.

In [ ]:
def calibration_curve_table(y, p, n_bins=10):
    df = pd.DataFrame({"y": y, "p": p})
    df["bucket"] = pd.qcut(df["p"], q=n_bins, duplicates="drop")
    out = (
        df
        .groupby("bucket", observed=False)
        .agg(
            n=("y", "count"),
            mean_pred_prob=("p", "mean"),
            realised_fill_rate=("y", "mean"),
        )
        .reset_index()
    )
    out["calibration_error"] = out["realised_fill_rate"] - out["mean_pred_prob"]
    return out

calibration_table = calibration_curve_table(y_test, test_prob, n_bins=10)

calibration_table

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
plt.scatter(calibration_table["mean_pred_prob"], calibration_table["realised_fill_rate"], s=70)
plt.title("Fill Probability Calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Realised fill rate")
plt.legend()
plt.show()

## 14. Feature importance

For logistic regression, signed coefficients indicate direction after scaling.

Interpret with care because correlated features can share explanatory power.

In [ ]:
def logistic_coefficients(model, feature_cols):
    if SKLEARN_AVAILABLE and isinstance(model, Pipeline):
        coef = model.named_steps["model"].coef_[0]
    else:
        coef = model.coef_[1:]

    return pd.DataFrame({
        "feature": feature_cols,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef),
    }).sort_values("abs_coefficient", ascending=False)

feature_importance = logistic_coefficients(logistic_model, feature_cols)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = feature_importance.sort_values("coefficient")
plt.barh(plot_df["feature"], plot_df["coefficient"])
plt.axvline(0, linestyle="--")
plt.title("Logistic Fill Model Coefficients")
plt.xlabel("Scaled coefficient")
plt.ylabel("Feature")
plt.show()

## 15. Discrete hazard model

For each order, create one row per event until fill or expiry.

Label:

$$
hazard_{i,k}=1
$$

if order $i$ fills at event bucket $k$, conditional on not filling earlier.

This approximates a survival model with logistic hazard.

In [ ]:
def build_hazard_dataset(model_df, stream, max_rows=250_000):
    rows = []
    rng = np.random.default_rng(config.seed + 555)

    # Sample orders if needed to keep notebook runtime friendly.
    orders_iter = model_df.copy()
    if len(orders_iter) > 5_000:
        orders_iter = orders_iter.sample(5_000, random_state=config.seed)

    for _, order in orders_iter.iterrows():
        active = int(order["active_event"])
        expiry = int(order["expiry_event"])
        fill_event = int(order["fill_event"])
        filled = int(order["filled"])

        for ev in range(active, expiry + 1):
            if len(rows) >= max_rows:
                break

            row = stream.iloc[ev]
            age = ev - active
            remaining = expiry - ev

            if filled and ev > fill_event:
                break

            hazard_y = int(filled and ev == fill_event)

            if order["side"] > 0:
                side_adj_flow = -row["flow_imbalance_50"]
                side_adj_book = -row["book_imbalance"]
            else:
                side_adj_flow = row["flow_imbalance_50"]
                side_adj_book = row["book_imbalance"]

            rows.append({
                "order_id": int(order["order_id"]),
                "event": ev,
                "age": age,
                "remaining_time": remaining,
                "side_buy": int(order["side"] > 0),
                "distance_ticks": order["distance_ticks"],
                "size_log": np.log1p(order["size"]),
                "queue_ahead_log": np.log1p(order["queue_ahead"]),
                "side_adjusted_flow_imbalance": side_adj_flow,
                "side_adjusted_book_imbalance": side_adj_book,
                "spread_ticks": row["spread_ticks"],
                "realised_vol_100": row["realised_vol_100"],
                "regime": row["regime"],
                "hazard_fill": hazard_y,
            })

        if len(rows) >= max_rows:
            break

    return pd.DataFrame(rows)

hazard_df = build_hazard_dataset(model_df, stream)

hazard_feature_cols = [
    "age",
    "remaining_time",
    "side_buy",
    "distance_ticks",
    "size_log",
    "queue_ahead_log",
    "side_adjusted_flow_imbalance",
    "side_adjusted_book_imbalance",
    "spread_ticks",
    "realised_vol_100",
    "regime",
]

hazard_df.head(), hazard_df["hazard_fill"].mean(), hazard_df.shape

## 16. Fit hazard model

In [ ]:
hazard_train, hazard_test = chronological_train_test(hazard_df, config.calibration_train_frac)

hazard_model = make_logistic_model()
hazard_model.fit(hazard_train[hazard_feature_cols].to_numpy(), hazard_train["hazard_fill"].to_numpy())

hazard_test_prob = hazard_model.predict_proba(hazard_test[hazard_feature_cols].to_numpy())[:, 1]
hazard_train_prob = hazard_model.predict_proba(hazard_train[hazard_feature_cols].to_numpy())[:, 1]

hazard_metrics = pd.DataFrame([
    {
        "sample": "train",
        "n": len(hazard_train),
        "hazard_rate": hazard_train["hazard_fill"].mean(),
        "auc": safe_auc(hazard_train["hazard_fill"].to_numpy(), hazard_train_prob),
        "brier": brier(hazard_train["hazard_fill"].to_numpy(), hazard_train_prob),
        "log_loss": safe_log_loss(hazard_train["hazard_fill"].to_numpy(), hazard_train_prob),
    },
    {
        "sample": "test",
        "n": len(hazard_test),
        "hazard_rate": hazard_test["hazard_fill"].mean(),
        "auc": safe_auc(hazard_test["hazard_fill"].to_numpy(), hazard_test_prob),
        "brier": brier(hazard_test["hazard_fill"].to_numpy(), hazard_test_prob),
        "log_loss": safe_log_loss(hazard_test["hazard_fill"].to_numpy(), hazard_test_prob),
    },
])

hazard_metrics

## 17. Convert hazard probabilities to fill-before-expiry probability

If per-step hazards are $h_k$, then:

$$
P(fill\ before\ expiry) = 1-\prod_k(1-h_k)
$$

We compare this survival-style probability with the direct logistic classifier.

In [ ]:
def predict_fill_prob_from_hazard(hazard_model, model_df_subset, stream, hazard_feature_cols):
    rows = []

    for _, order in model_df_subset.iterrows():
        active = int(order["active_event"])
        expiry = int(order["expiry_event"])

        feature_rows = []
        for ev in range(active, expiry + 1):
            row = stream.iloc[ev]
            age = ev - active
            remaining = expiry - ev

            if order["side"] > 0:
                side_adj_flow = -row["flow_imbalance_50"]
                side_adj_book = -row["book_imbalance"]
            else:
                side_adj_flow = row["flow_imbalance_50"]
                side_adj_book = row["book_imbalance"]

            feature_rows.append({
                "age": age,
                "remaining_time": remaining,
                "side_buy": int(order["side"] > 0),
                "distance_ticks": order["distance_ticks"],
                "size_log": np.log1p(order["size"]),
                "queue_ahead_log": np.log1p(order["queue_ahead"]),
                "side_adjusted_flow_imbalance": side_adj_flow,
                "side_adjusted_book_imbalance": side_adj_book,
                "spread_ticks": row["spread_ticks"],
                "realised_vol_100": row["realised_vol_100"],
                "regime": row["regime"],
            })

        X = pd.DataFrame(feature_rows)[hazard_feature_cols].to_numpy()
        h = hazard_model.predict_proba(X)[:, 1]
        fill_prob = 1.0 - np.prod(1.0 - np.clip(h, 0, 1))

        rows.append({
            "order_id": int(order["order_id"]),
            "hazard_fill_prob": fill_prob,
        })

    return pd.DataFrame(rows)

hazard_order_probs = predict_fill_prob_from_hazard(hazard_model, test_df, stream, hazard_feature_cols)

combined_prob = test_predictions.merge(hazard_order_probs, on="order_id", how="left")
combined_prob["filled"] = combined_prob["filled"].astype(int)

combined_metrics = pd.DataFrame([
    {
        "model": "direct_logistic",
        "auc": safe_auc(combined_prob["filled"].to_numpy(), combined_prob["pred_fill_prob"].to_numpy()),
        "brier": brier(combined_prob["filled"].to_numpy(), combined_prob["pred_fill_prob"].to_numpy()),
        "log_loss": safe_log_loss(combined_prob["filled"].to_numpy(), combined_prob["pred_fill_prob"].to_numpy()),
    },
    {
        "model": "hazard_survival",
        "auc": safe_auc(combined_prob["filled"].to_numpy(), combined_prob["hazard_fill_prob"].to_numpy()),
        "brier": brier(combined_prob["filled"].to_numpy(), combined_prob["hazard_fill_prob"].to_numpy()),
        "log_loss": safe_log_loss(combined_prob["filled"].to_numpy(), combined_prob["hazard_fill_prob"].to_numpy()),
    },
])

combined_metrics

## 18. Passive versus marketable expected value

A passive order has:

$$
\begin{aligned}
EV_{passive} &= p_{fill} ( spreadCapture \\
&\quad - adverseSelection \\
&\quad + rebate ) \\
&\quad - (1-p_{fill})opportunityCost
\end{aligned}
$$

A marketable order has:

$$
\begin{aligned}
EV_{cross} &= -spreadCost \\
&\quad - fee
\end{aligned}
$$

The model chooses passive only when the expected value is favourable and fill probability is high enough.

In [ ]:
def passive_vs_cross_decision(test_df, predicted_prob, config):
    df = test_df.copy()
    df["pred_fill_prob"] = predicted_prob

    # Estimate passive edge using observed feature proxies.
    half_spread_bps = df["active_spread_ticks"] * config.tick_size / df["active_mid"] * 10000 / 2.0

    expected_spread_capture_bps = half_spread_bps
    expected_adverse_bps = np.maximum(0.0, 2.0 * df["active_realised_vol_100"] * 10000)
    opportunity_cost_bps = half_spread_bps * 0.50

    df["passive_ev_bps"] = (
        df["pred_fill_prob"] * (expected_spread_capture_bps - expected_adverse_bps - config.passive_rebate_bps)
        - (1.0 - df["pred_fill_prob"]) * opportunity_cost_bps
    )

    df["cross_ev_bps"] = -half_spread_bps - config.taker_fee_bps
    df["choose_passive"] = (df["passive_ev_bps"] > df["cross_ev_bps"]) & (df["pred_fill_prob"] >= config.probability_threshold)

    # Realised passive value only where filled; otherwise use opportunity cost proxy.
    realised_passive_bps = np.where(
        df["filled"] == 1,
        df["net_edge_bps"].fillna(0.0),
        -opportunity_cost_bps,
    )
    df["realised_policy_bps"] = np.where(df["choose_passive"], realised_passive_bps, df["cross_ev_bps"])

    return df

decision_df = passive_vs_cross_decision(test_df, test_prob, config)

decision_summary = pd.DataFrame([{
    "n_test_orders": len(decision_df),
    "choose_passive_rate": decision_df["choose_passive"].mean(),
    "mean_pred_fill_prob_passive_chosen": decision_df.loc[decision_df["choose_passive"], "pred_fill_prob"].mean(),
    "realised_mean_policy_bps": decision_df["realised_policy_bps"].mean(),
    "realised_mean_if_always_passive_bps": np.where(decision_df["filled"] == 1, decision_df["net_edge_bps"].fillna(0.0), -decision_df["active_spread_ticks"] * config.tick_size / decision_df["active_mid"] * 10000 / 4.0).mean(),
    "mean_cross_ev_bps": decision_df["cross_ev_bps"].mean(),
}])

decision_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(decision_df["passive_ev_bps"], bins=60, alpha=0.7, label="passive EV")
plt.hist(decision_df["cross_ev_bps"], bins=60, alpha=0.7, label="cross EV")
plt.axvline(0, linestyle="--")
plt.title("Passive vs Cross Expected Value")
plt.xlabel("bps")
plt.ylabel("Count")
plt.legend()
plt.show()

## 19. Threshold policy sensitivity

Vary the minimum fill probability required before choosing passive.

In [ ]:
threshold_rows = []

for threshold in np.linspace(0.10, 0.90, 17):
    cfg = LimitOrderFillConfig(
        n_events=config.n_events,
        n_orders=config.n_orders,
        seed=config.seed,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        base_spread_ticks=config.base_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        base_depth=config.base_depth,
        min_time_in_force_events=config.min_time_in_force_events,
        max_time_in_force_events=config.max_time_in_force_events,
        max_latency_events=config.max_latency_events,
        passive_rebate_bps=config.passive_rebate_bps,
        taker_fee_bps=config.taker_fee_bps,
        adverse_selection_horizon=config.adverse_selection_horizon,
        order_size_median=config.order_size_median,
        calibration_train_frac=config.calibration_train_frac,
        probability_threshold=float(threshold),
        output_subdir=config.output_subdir,
    )

    d = passive_vs_cross_decision(test_df, test_prob, cfg)
    threshold_rows.append({
        "threshold": threshold,
        "choose_passive_rate": d["choose_passive"].mean(),
        "realised_mean_policy_bps": d["realised_policy_bps"].mean(),
        "passive_fill_rate_when_chosen": d.loc[d["choose_passive"], "filled"].mean() if d["choose_passive"].any() else np.nan,
    })

threshold_sensitivity = pd.DataFrame(threshold_rows)

threshold_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(threshold_sensitivity["threshold"], threshold_sensitivity["realised_mean_policy_bps"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Realised Policy Value vs Fill-Probability Threshold")
plt.xlabel("Probability threshold")
plt.ylabel("Realised mean policy bps")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(threshold_sensitivity["threshold"], threshold_sensitivity["choose_passive_rate"], marker="o")
plt.title("Passive Usage vs Probability Threshold")
plt.xlabel("Probability threshold")
plt.ylabel("Choose passive rate")
plt.show()

## 20. Latency sensitivity

Latency changes the active quote state and queue position.

We regenerate orders with different maximum latency and evaluate model robustness.

In [ ]:
latency_rows = []

for max_latency in [0, 1, 2, 4, 8, 12, 20]:
    cfg = LimitOrderFillConfig(
        n_events=config.n_events,
        n_orders=5_000,
        seed=config.seed + max_latency,
        initial_mid=config.initial_mid,
        tick_size=config.tick_size,
        base_spread_ticks=config.base_spread_ticks,
        max_spread_ticks=config.max_spread_ticks,
        base_depth=config.base_depth,
        min_time_in_force_events=config.min_time_in_force_events,
        max_time_in_force_events=config.max_time_in_force_events,
        max_latency_events=max_latency,
        passive_rebate_bps=config.passive_rebate_bps,
        taker_fee_bps=config.taker_fee_bps,
        adverse_selection_horizon=config.adverse_selection_horizon,
        order_size_median=config.order_size_median,
        calibration_train_frac=config.calibration_train_frac,
        probability_threshold=config.probability_threshold,
        output_subdir=config.output_subdir,
    )

    o = generate_candidate_limit_orders(stream, cfg)
    out = simulate_limit_order_outcomes(stream, o, cfg)
    df, fcols = build_model_dataset(o, out)

    if len(df) == 0:
        continue

    # Score using base model on common feature set.
    p = logistic_model.predict_proba(df[feature_cols].to_numpy())[:, 1]
    latency_rows.append({
        "max_latency_events": max_latency,
        "n_orders": len(df),
        "realised_fill_rate": df["filled"].mean(),
        "mean_pred_fill_prob": p.mean(),
        "auc": safe_auc(df["filled"].to_numpy(), p),
        "brier": brier(df["filled"].to_numpy(), p),
    })

latency_sensitivity = pd.DataFrame(latency_rows)

latency_sensitivity

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(latency_sensitivity["max_latency_events"], latency_sensitivity["realised_fill_rate"], marker="o", label="realised")
plt.plot(latency_sensitivity["max_latency_events"], latency_sensitivity["mean_pred_fill_prob"], marker="o", label="predicted")
plt.title("Fill Probability vs Latency")
plt.xlabel("Max latency events")
plt.ylabel("Fill rate / predicted probability")
plt.legend()
plt.show()

## 21. Adverse-selection diagnostics after fill

A fill can be bad if price moves against the passive order afterwards.

We bucket realised adverse selection by predicted fill probability.

In [ ]:
adverse_bucket = decision_df.copy()
adverse_bucket["prob_bucket"] = pd.qcut(adverse_bucket["pred_fill_prob"], q=10, duplicates="drop")

adverse_selection_report = (
    adverse_bucket
    .groupby("prob_bucket", observed=False)
    .agg(
        n=("filled", "count"),
        mean_pred_prob=("pred_fill_prob", "mean"),
        realised_fill_rate=("filled", "mean"),
        mean_adverse_selection_bps=("adverse_selection_bps", "mean"),
        mean_net_edge_bps=("net_edge_bps", "mean"),
        choose_passive_rate=("choose_passive", "mean"),
    )
    .reset_index()
)

adverse_selection_report

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(adverse_selection_report["mean_pred_prob"], adverse_selection_report["mean_net_edge_bps"], s=80)
plt.axhline(0, linestyle="--")
plt.title("Net Edge by Predicted Fill Probability Bucket")
plt.xlabel("Mean predicted fill probability")
plt.ylabel("Mean net edge, bps")
plt.show()

## 22. Governance flags

A limit-order fill model should be reviewed if:

1. AUC is weak;
2. calibration error is high;
3. Brier score is poor;
4. hazard model disagrees strongly with direct model;
5. passive decision policy loses money;
6. latency sensitivity is high;
7. adverse selection dominates spread capture;
8. synthetic data is used instead of real order/fill data.

In [ ]:
test_metric = model_metrics[model_metrics["sample"] == "test"].iloc[0]
calibration_mae = calibration_table["calibration_error"].abs().mean()
hazard_brier = combined_metrics[combined_metrics["model"] == "hazard_survival"]["brier"].iloc[0]
direct_brier = combined_metrics[combined_metrics["model"] == "direct_logistic"]["brier"].iloc[0]
hazard_direct_brier_gap = hazard_brier - direct_brier
policy_value = decision_summary["realised_mean_policy_bps"].iloc[0]

latency_fill_range = latency_sensitivity["realised_fill_rate"].max() - latency_sensitivity["realised_fill_rate"].min()
mean_adverse = decision_df.loc[decision_df["filled"] == 1, "adverse_selection_bps"].mean()
mean_spread_capture = decision_df.loc[decision_df["filled"] == 1, "spread_capture_bps"].mean()

governance_flags = pd.DataFrame([{
    "test_auc": test_metric["auc"],
    "test_brier": test_metric["brier"],
    "test_log_loss": test_metric["log_loss"],
    "calibration_mae": calibration_mae,
    "hazard_direct_brier_gap": hazard_direct_brier_gap,
    "realised_policy_value_bps": policy_value,
    "latency_fill_rate_range": latency_fill_range,
    "mean_adverse_selection_bps_when_filled": mean_adverse,
    "mean_spread_capture_bps_when_filled": mean_spread_capture,
    "flag_auc_weak": bool(test_metric["auc"] < 0.60),
    "flag_calibration_error_high": bool(calibration_mae > 0.08),
    "flag_brier_high": bool(test_metric["brier"] > 0.25),
    "flag_hazard_disagrees": bool(abs(hazard_direct_brier_gap) > 0.05),
    "flag_passive_policy_loses_money": bool(policy_value < 0),
    "flag_latency_sensitivity_high": bool(latency_fill_range > 0.25),
    "flag_adverse_selection_dominates": bool(mean_adverse > mean_spread_capture),
    "flag_synthetic_data_not_real_fills": True,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_auc_weak",
        "flag_calibration_error_high",
        "flag_brier_high",
        "flag_hazard_disagrees",
        "flag_passive_policy_loses_money",
        "flag_latency_sensitivity_high",
        "flag_adverse_selection_dominates",
        "flag_synthetic_data_not_real_fills",
    ]
].any(axis=1)

governance_flags

## 23. Audit checks

Numerical and process checks:

1. target labels are binary;
2. predicted probabilities are in $[0,1]$;
3. train/test split is chronological;
4. no feature is non-finite;
5. calibration table is non-empty;
6. hazard probabilities are valid;
7. decision policy outputs are finite;
8. governance flags exist.

In [ ]:
audit_rows = []

labels_binary = bool(set(model_df["filled"].unique()).issubset({0, 1}))
audit_rows.append({
    "check": "labels_binary",
    "value": labels_binary,
    "passed": labels_binary,
})

prob_valid = bool(((test_prob >= 0) & (test_prob <= 1)).all())
audit_rows.append({
    "check": "direct_model_probabilities_valid",
    "value": prob_valid,
    "passed": prob_valid,
})

chronological = bool(train_df["active_event"].max() <= test_df["active_event"].min())
audit_rows.append({
    "check": "train_test_chronological",
    "value": chronological,
    "passed": chronological,
})

features_finite = bool(np.isfinite(model_df[feature_cols].to_numpy()).all())
audit_rows.append({
    "check": "features_finite",
    "value": features_finite,
    "passed": features_finite,
})

calibration_nonempty = bool(len(calibration_table) > 0)
audit_rows.append({
    "check": "calibration_table_nonempty",
    "value": calibration_nonempty,
    "passed": calibration_nonempty,
})

hazard_prob_valid = bool(((combined_prob["hazard_fill_prob"] >= 0) & (combined_prob["hazard_fill_prob"] <= 1)).all())
audit_rows.append({
    "check": "hazard_fill_probabilities_valid",
    "value": hazard_prob_valid,
    "passed": hazard_prob_valid,
})

decision_finite = bool(np.isfinite(decision_df[["passive_ev_bps", "cross_ev_bps", "realised_policy_bps"]].to_numpy()).all())
audit_rows.append({
    "check": "decision_policy_outputs_finite",
    "value": decision_finite,
    "passed": decision_finite,
})

governance_exists = bool(len(governance_flags) == 1)
audit_rows.append({
    "check": "governance_flags_exist",
    "value": governance_exists,
    "passed": governance_exists,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 24. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed") / config.output_subdir
output_dir.mkdir(parents=True, exist_ok=True)

stream.to_csv(output_dir / "market_stream.csv", index=False)
orders.to_csv(output_dir / "candidate_orders.csv", index=False)
outcomes.to_csv(output_dir / "order_outcomes.csv", index=False)
model_df.to_csv(output_dir / "model_dataset.csv", index=False)
fill_bucket_report.to_csv(output_dir / "fill_bucket_report.csv", index=False)
train_df.to_csv(output_dir / "train_dataset.csv", index=False)
test_df.to_csv(output_dir / "test_dataset.csv", index=False)
test_predictions.to_csv(output_dir / "test_predictions_direct.csv", index=False)
model_metrics.to_csv(output_dir / "model_metrics.csv", index=False)
calibration_table.to_csv(output_dir / "calibration_table.csv", index=False)
feature_importance.to_csv(output_dir / "feature_importance.csv", index=False)
hazard_df.to_csv(output_dir / "hazard_dataset.csv", index=False)
hazard_metrics.to_csv(output_dir / "hazard_metrics.csv", index=False)
combined_prob.to_csv(output_dir / "combined_direct_hazard_probabilities.csv", index=False)
combined_metrics.to_csv(output_dir / "combined_model_metrics.csv", index=False)
decision_df.to_csv(output_dir / "passive_vs_cross_decision.csv", index=False)
decision_summary.to_csv(output_dir / "decision_summary.csv", index=False)
threshold_sensitivity.to_csv(output_dir / "threshold_sensitivity.csv", index=False)
latency_sensitivity.to_csv(output_dir / "latency_sensitivity.csv", index=False)
adverse_selection_report.to_csv(output_dir / "adverse_selection_report.csv", index=False)
governance_flags.to_csv(output_dir / "governance_flags.csv", index=False)
audit_report.to_csv(output_dir / "audit_report.csv", index=False)

manifest = {
    "dataset_name": "limit_order_fill_probability_model_outputs",
    "schema_version": "limit_order_fill_probability_model_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "sklearn_available": SKLEARN_AVAILABLE,
    "n_orders": int(len(orders)),
    "fill_rate": float(model_df["filled"].mean()),
    "feature_cols": feature_cols,
    "hazard_feature_cols": hazard_feature_cols,
    "model_metrics": model_metrics.to_dict(orient="records"),
    "combined_metrics": combined_metrics.to_dict(orient="records"),
    "decision_summary": decision_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "This notebook models limit-order fill probability using synthetic top-of-book and order-flow features.",
        "Direct logistic and discrete hazard-style fill models are compared.",
        "Features include queue proxy, distance to touch, time-in-force, latency, imbalance, spread, volatility and regime.",
        "Calibration curves, Brier score, AUC and log loss are reported.",
        "A passive-versus-cross expected-value policy is demonstrated.",
        "Latency and adverse-selection diagnostics are included.",
        "The data is synthetic and should be replaced with real order, quote and fill records before production use."
    ],
}

manifest_path = output_dir / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

output_dir / "model_metrics.csv", output_dir / "decision_summary.csv", output_dir / "governance_flags.csv", manifest_path

## 25. Practical checklist for limit-order fill models

Before using a fill probability model:

1. **Use real order and fill records.**
2. **Timestamp quotes and orders precisely.**
3. **Account for latency.**
4. **Use queue-position features when possible.**
5. **Separate passive, marketable-limit and market orders.**
6. **Measure time-to-fill, not just fill/no-fill.**
7. **Check probability calibration.**
8. **Include adverse selection after fill.**
9. **Use chronological validation.**
10. **Do not optimise only for fill rate; optimise for fill quality.**

## 26. Limitations

### Synthetic data

The notebook uses synthetic quote, order and fill data.

### Queue proxy

Queue ahead is approximated from displayed depth. Real queue position requires order acknowledgements, cancellations and trades.

### No full L2 book

The model uses L1-like features and synthetic flow, not full order book depth.

### Simplified adverse selection

Adverse selection is measured by future mid-price movement over a fixed horizon.

### No cancellation optimisation

The model scores candidate orders but does not optimise cancel/replace timing.

### No exchange-specific matching rules

Real matching depends on price-time priority, pro-rata rules, auctions, hidden orders and venue-specific behaviour.

## 27. Research frontier and extensions

Important extensions include:

1. queue-position survival models;
2. Cox proportional hazard models;
3. recurrent neural point-process fill models;
4. competing-risk models for fill/cancel/adverse move;
5. L2 depth features;
6. order-age and queue-depletion models;
7. microprice and imbalance features;
8. Hawkes-driven fill intensities;
9. venue-specific matching rules;
10. online recalibration from live fills.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **06_10_execution_risk_controls_and_kill_switch**  
   Use fill probability and adverse selection to block bad orders.

2. **06_11_l1_bid_ask_backtest_execution_model**  
   Integrate fill probabilities into strategy backtesting.

3. **06_12_production_reconciliation_and_breaks**  
   Compare predicted fill probabilities with actual fills.

4. **06_13_execution_research_capstone**  
   Combine impact, fill probability, market making and execution controls.

## 29. Summary

This notebook implemented a limit-order fill probability model.

It showed how to:

1. simulate L1-like market state;
2. generate candidate passive limit orders;
3. simulate fill outcomes;
4. build fill-model features;
5. estimate direct logistic fill probabilities;
6. evaluate AUC, Brier score and log loss;
7. build calibration curves;
8. inspect feature coefficients;
9. construct a discrete hazard dataset;
10. estimate hazard-style fill probabilities;
11. compare direct and hazard models;
12. evaluate passive versus marketable expected value;
13. test threshold policies;
14. analyse latency sensitivity;
15. analyse adverse selection after fill;
16. create governance flags and audit checks;
17. save outputs and manifests.

The key computational takeaway:

> Limit-order fill probability can be treated as a calibrated probabilistic prediction problem using quote, queue, imbalance, latency and time-in-force features.

The key financial takeaway:

> A high fill rate is not automatically good; the best passive orders are the ones that fill without being adversely selected.

## 30. Further reading

- Harris, *Trading and Exchanges.*
- Cartea, Jaimungal and Penalva, *Algorithmic and High-Frequency Trading.*
- Abergel et al., *Limit Order Books.*
- Guéant, *The Financial Mathematics of Market Liquidity.*
- Cont and de Larrard on price dynamics in limit order books.
- Queue-reactive and survival models for limit order fill probabilities.